# Sentiment Analysis Model Evaluation: Logistic Regression

## Understanding False Positives
In binary classification for sentiment analysis:
- **False Positive (FP)**: The model predicts positive sentiment (e.g., "good" or "happy") when the actual sentiment is negative.
- This means the model is incorrectly classifying negative reviews as positive.

## Confusion Matrix
A confusion matrix shows:
- True Positive (TP): Correctly predicted positive
- True Negative (TN): Correctly predicted negative  
- False Positive (FP): Incorrectly predicted positive
- False Negative (FN): Incorrectly predicted negative

## Reducing False Positives
To reduce false positives, we can:
1. **Increase the prediction threshold**: Instead of 0.5, use 0.6. This makes the model more conservative about predicting positive sentiment.
2. **Adjust learning rate in gradient descent**: During training, a different learning rate can affect model convergence and potentially reduce overfitting that leads to false positives.

Let's implement this with our logistic regression model.

In [ ]:
# Import necessary libraries
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Load the trained Logistic Regression model
with open('models/LogisticRegression.pkl', 'rb') as f:
    model = pickle.load(f)

# Load the vectorizer
with open('features/vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

print("Model and vectorizer loaded successfully!")

# Load dataset for testing
data = pd.read_csv('dataset/Womens Clothing E-Commerce Reviews.csv')
data = data.dropna(subset=['Review Text'])  # Remove rows with missing reviews
data['sentiment'] = data['Rating'].apply(lambda x: 1 if x >= 4 else 0)  # Positive if rating >=4

print(f"Dataset shape: {data.shape}")
print("Sentiment distribution:")
print(data['sentiment'].value_counts())

# Split data for testing (using same split as training if possible, but for demo, random split)
X = data['Review Text']
y = data['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transform test data
X_test_tfidf = vectorizer.transform(X_test)

print(f"Test set shape: {X_test_tfidf.shape}")

In [ ]:
# Predict with default threshold 0.5
y_pred_prob = model.predict_proba(X_test_tfidf)[:, 1]  # Probability of positive class
y_pred_default = (y_pred_prob >= 0.5).astype(int)

# Confusion Matrix
cm_default = confusion_matrix(y_test, y_pred_default)
print("Confusion Matrix (Threshold = 0.5):")
print(cm_default)

# Extract FP
tn, fp, fn, tp = cm_default.ravel()
print(f"True Negative: {tn}, False Positive: {fp}, False Negative: {fn}, True Positive: {tp}")
print(f"False Positive Rate: {fp / (fp + tn):.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_default))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm_default, display_labels=['Negative', 'Positive'])
disp.plot()
plt.title('Confusion Matrix (Threshold = 0.5)')
plt.show()

In [ ]:
# Predict with threshold 0.6 to reduce false positives
y_pred_06 = (y_pred_prob >= 0.6).astype(int)

# Confusion Matrix
cm_06 = confusion_matrix(y_test, y_pred_06)
print("Confusion Matrix (Threshold = 0.6):")
print(cm_06)

# Extract FP
tn, fp, fn, tp = cm_06.ravel()
print(f"True Negative: {tn}, False Positive: {fp}, False Negative: {fn}, True Positive: {tp}")
print(f"False Positive Rate: {fp / (fp + tn):.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_06))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm_06, display_labels=['Negative', 'Positive'])
disp.plot()
plt.title('Confusion Matrix (Threshold = 0.6)')
plt.show()

print("By increasing the threshold to 0.6, we reduced false positives from the previous value.")

## Gradient Descent and Learning Rate

In logistic regression, the model is trained using gradient descent to minimize the loss function. The learning rate controls how much the model parameters are updated in each iteration.

- **High learning rate**: Faster convergence but may overshoot the minimum, leading to instability.
- **Low learning rate**: Slower convergence but more stable, may get stuck in local minima.

In scikit-learn's LogisticRegression, the learning rate is handled internally by the solver. Different solvers use different optimization algorithms:
- 'lbfgs': Quasi-Newton method
- 'liblinear': Coordinate descent
- 'newton-cg': Newton method

To demonstrate the effect, let's retrain the model with a different solver and compare performance.

In [ ]:
# Retrain model with different solver (simulating different learning rate behavior)
from sklearn.linear_model import LogisticRegression

# Original model used default solver (lbfgs)
# Let's try with liblinear solver, which uses coordinate descent
model_new = LogisticRegression(solver='liblinear', max_iter=1000)
X_train_tfidf = vectorizer.transform(X_train)
model_new.fit(X_train_tfidf, y_train)

# Predict with new model
y_pred_prob_new = model_new.predict_proba(X_test_tfidf)[:, 1]
y_pred_new = (y_pred_prob_new >= 0.5).astype(int)

# Confusion Matrix for new model
cm_new = confusion_matrix(y_test, y_pred_new)
print("Confusion Matrix (New Model with liblinear solver):")
print(cm_new)

tn, fp, fn, tp = cm_new.ravel()
print(f"False Positive Rate: {fp / (fp + tn):.4f}")

print("\nComparison:")
print("Original model FP rate vs New model FP rate")
# (We would compare, but since it's demo, just show)

# Also, show the effect of threshold on new model
y_pred_new_06 = (y_pred_prob_new >= 0.6).astype(int)
cm_new_06 = confusion_matrix(y_test, y_pred_new_06)
print("\nConfusion Matrix (New Model, Threshold = 0.6):")
print(cm_new_06)

tn, fp, fn, tp = cm_new_06.ravel()
print(f"False Positive Rate: {fp / (fp + tn):.4f}")